# DEKM 2021 20NEWS official-source matrix, seeds 42-44

Seed 42 established that the official DEKM `utils.py` source path - `fetch_20newsgroups(subset="all")` plus `TfidfVectorizer(max_features=2000)` - restores the paper count of 18,846 rows and materially improves ACC relative to the unrelated 18,057-row Google Drive NPZ files. This notebook now runs seeds 42-44 under that immutable source-faithful contract and reports mean plus sample standard deviation (`ddof=1`).


In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
print('TensorFlow:', tf.__version__)
print('GPUs:', gpus)
assert gpus, 'Enable a GPU accelerator in Kaggle notebook settings.'


In [ ]:
from pathlib import Path
import subprocess
import sys

REPO = Path('/kaggle/working/mvdec_fpmax')
subprocess.run(['rm', '-rf', str(REPO)], check=True)
subprocess.run(['git', 'clone', '--branch', 'dev', '--single-branch', 'https://github.com/vdat-18/mvdec_fpmax.git', str(REPO)], check=True)
EXPECTED_REVISION = 'f3aa64c0e976356964baafb8b26f57c6d92eead4'
subprocess.run(['git', 'checkout', '--detach', EXPECTED_REVISION], cwd=REPO, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
if str(REPO / 'src') not in sys.path:
    sys.path.insert(0, str(REPO / 'src'))
actual = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip()
assert actual == EXPECTED_REVISION, actual
print('Repository revision:', actual)


In [ ]:
import json
import sklearn
from sklearn.datasets import fetch_20newsgroups

raw = fetch_20newsgroups(subset='all', download_if_missing=True)
assert len(raw.data) == 18846, len(raw.data)
class_counts = {str(label): int((raw.target == label).sum()) for label in range(20)}
source_audit = {
    'source_id': 'sklearn_fetch_20newsgroups_all_tfidf2000_18846',
    'raw_rows': len(raw.data),
    'class_counts': class_counts,
    'sklearn_version': sklearn.__version__,
}
OUTPUT = Path('/kaggle/working/dekm-2021-20news-sklearn18846-seeds42-44')
RUN_ROOT = OUTPUT / 'runs'
OUTPUT.mkdir(parents=True, exist_ok=True)
(OUTPUT / 'source_audit.json').write_text(json.dumps(source_audit, indent=2), encoding='utf-8')
print(json.dumps(source_audit, indent=2))

command = [
    sys.executable, '-m', 'representation_learning.DEKM_dense', '20NEWS',
    '--protocol', 'release-2021-sklearn-20news', '--seed', '42', '--runs', '3',
    '--require-gpu', '--dataset-root', str(REPO / 'external_repos/DEKM/datasets'),
    '--output-dir', str(RUN_ROOT),
]
subprocess.run(command, cwd=REPO, check=True)


In [ ]:
import hashlib
import pandas as pd
from pipeline.external_metrics import compute_external_metrics

PROTOCOL = 'dekm_2021_20news_sklearn18846_seeded_modern_runtime_v1'
SEEDS = (42, 43, 44)
protocol_root = RUN_ROOT / '20news' / PROTOCOL
config_roots = [path for path in protocol_root.iterdir() if path.is_dir()]
assert len(config_roots) == 1, config_roots
config_root = config_roots[0]
runs = pd.read_csv(config_root / 'dekm_runs.csv').sort_values('seed')
assert runs['seed'].tolist() == list(SEEDS)
assert runs['n_samples'].tolist() == [18846] * 3
assert runs['source_id'].nunique() == 1
assert runs['source_id'].iloc[0] == 'sklearn_fetch_20newsgroups_all_tfidf2000_18846'

for seed in SEEDS:
    run_dir = config_root / f'seed_{seed}'
    manifest = json.loads((run_dir / 'manifest.json').read_text(encoding='utf-8'))
    assert manifest['status'] == 'complete'
    assert manifest['protocol_id'] == PROTOCOL
    assert manifest['source_id'] == 'sklearn_fetch_20newsgroups_all_tfidf2000_18846'
    assert manifest['n_samples'] == 18846
    assert manifest['config']['refinement_batch_size'] == 256
    assert manifest['config']['assignment_change_tolerance'] == 0.005
    assert manifest['config']['input_normalization'] == 'release_l2_unit_norm'
    assert manifest['official_repository_paper_era_revision'] == '129b7d08b545f02d920ccff0948ec44b654b3158'
    assert manifest['official_repository_paper_era_utils_sha256'] == '93b18e8b09559b2ddd74fe631e44670ead935e7dd962502e5aa0d0d2b22f4451'
    assignments = pd.read_csv(run_dir / 'assignments.csv')
    metrics = compute_external_metrics(assignments['true_label'].to_numpy(), assignments['cluster'].to_numpy(), n_clusters=20)
    assert abs(metrics.acc - manifest['metrics']['acc']) < 1e-12
    assert abs(metrics.nmi - manifest['metrics']['nmi']) < 1e-12
    for name, filename in manifest['paths'].items():
        path = run_dir / filename
        assert hashlib.sha256(path.read_bytes()).hexdigest() == manifest['output_sha256'][name]

paper = {'acc': 0.4108, 'nmi': 0.4027}
aggregate = {
    'dataset': '20NEWS', 'protocol_id': PROTOCOL, 'completed_runs': 3,
    'acc_mean': float(runs['acc'].mean()), 'acc_sample_std': float(runs['acc'].std(ddof=1)),
    'nmi_mean': float(runs['nmi'].mean()), 'nmi_sample_std': float(runs['nmi'].std(ddof=1)),
    'runtime_seconds_mean': float(runs['fit_time_seconds'].mean()),
    'paper_acc': paper['acc'], 'paper_nmi': paper['nmi'],
    'acc_delta_pp': float(100 * (runs['acc'].mean() - paper['acc'])),
    'nmi_delta_pp': float(100 * (runs['nmi'].mean() - paper['nmi'])),
    'best_acc_seed': int(runs.loc[runs['acc'].idxmax(), 'seed']),
    'best_acc': float(runs['acc'].max()),
    'best_nmi_seed': int(runs.loc[runs['nmi'].idxmax(), 'seed']),
    'best_nmi': float(runs['nmi'].max()),
    'source_id': runs['source_id'].iloc[0], 'source_sha256': runs['source_sha256'].iloc[0],
    'config_hash': runs['config_hash'].iloc[0],
}
runs.to_csv(OUTPUT / 'dekm_runs.csv', index=False)
pd.DataFrame([aggregate]).to_csv(OUTPUT / 'dekm_aggregate.csv', index=False)
(OUTPUT / 'dekm_aggregate.json').write_text(json.dumps(aggregate, indent=2), encoding='utf-8')
(OUTPUT / 'failures.json').write_text('[]\n', encoding='utf-8')
display(runs[['seed', 'acc', 'nmi', 'fit_time_seconds', 'stop_reason']])
display(pd.DataFrame([aggregate]))


In [ ]:
import shutil

zip_path = Path(shutil.make_archive('/kaggle/working/dekm-2021-20news-sklearn18846-seeds42-44', 'zip', '/kaggle/working', 'dekm-2021-20news-sklearn18846-seeds42-44'))
digest = hashlib.sha256(zip_path.read_bytes()).hexdigest()
sha_path = zip_path.with_suffix('.zip.sha256')
sha_path.write_text(f'{digest}  {zip_path.name}\n', encoding='ascii')
print('ZIP:', zip_path)
print('SHA-256:', digest)
